# Pré-processamento 1 — Arapiraca

Este notebook realiza:
1. Carregamento do CSV filtrado
2. Obtenção do polígono municipal via OpenStreetMap
3. Cálculo dinâmico do tamanho do grid
4. Geração da máscara de validade (células dentro do polígono)
5. Filtragem espacial dos pontos de crime
6. Visualização em mapa interativo

**Entrada:** `./output/arapiraca/filtered_arapiraca.csv`

**Saída:** `osm_arapiraca.geojson`, `arapiraca_mask_final.npy`, `filtered_arapiraca_inside_polygon.csv`

## 1. Carregamento dos dados filtrados de Arapiraca

In [6]:
# Carrega o CSV de Arapiraca gerado pelo notebook pre-processing.ipynb

import pandas as pd

df_arapiraca = pd.read_csv(
    "./output/arapiraca/filtered_arapiraca.csv", delimiter=";", low_memory=False
)

In [7]:
display(df_arapiraca)

,LONGITUDE,LATITUDE,ROUBO_GROUP,DATA_HORA,CIDADE_FATO
0,-36.690147,-9.735983,street_robbery,2022-01-01 03:00:00,Arapiraca
1,-36.655517,-9.785501,street_robbery,2022-01-01 20:00:00,Arapiraca
2,-36.591520,-9.762967,vehicle_robbery,2022-01-02 09:00:00,Arapiraca
3,-36.664139,-9.768197,street_robbery,2022-01-02 23:00:00,Arapiraca
4,-36.680604,-9.644294,street_robbery,2022-01-03 08:00:00,Arapiraca
...,...,...,...,...,...
17416,-36.662065,-9.764001,vehicle_robbery,2014-12-21 01:30:00,Arapiraca
17417,-36.666260,-9.753766,street_robbery,2014-12-22 14:30:00,Arapiraca
17418,-36.663400,-9.748675,street_robbery,2014-12-22 22:00:00,Arapiraca
17419,-36.648386,-9.721165,vehicle_robbery,2014-12-22 23:30:00,Arapiraca


## 2. Obtenção do polígono municipal via OpenStreetMap

Consulta o OpenStreetMap para obter o polígono dos limites de Arapiraca. Seleciona o maior polígono encontrado e salva em GeoJSON.

In [8]:
# Busca o polígono de Arapiraca no OpenStreetMap
# Seleciona o maior polígono (por área) entre os resultados
# Salva em formato GeoJSON para uso nas etapas seguintes

import osmnx as ox
import geopandas as gpd

place = "Arapiraca, Alagoas, Brazil"
print(f"Querying OSM for: {place}")

gdf_place = ox.geocode_to_gdf(place)
polys = gdf_place[gdf_place.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    # fallback: fetch administrative boundaries and pick candidates
    tags = {"boundary": "administrative"}
    adm = ox.geometries_from_place(place, tags=tags)
    polys = adm[adm.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()

if polys.empty:
    raise SystemExit(
        "Could not find polygon for Arapiraca on OSM. Check query or network."
    )

# # compute area in metric projection to pick the largest polygon
polys = polys.to_crs("EPSG:3857")
polys["area_m2"] = polys.geometry.area
largest_idx = polys["area_m2"].idxmax()
poly = polys.loc[largest_idx].geometry
# convert back to WGS84
poly = gpd.GeoSeries([poly], crs="EPSG:3857").to_crs("EPSG:4326").iloc[0]

osm_arapiraca = gpd.GeoDataFrame({"name": ["Arapiraca"], "geometry": [poly]}, crs="EPSG:4326")
osm_path = "./output/arapiraca/osm_arapiraca.geojson"
osm_arapiraca.to_file(osm_path, driver="GeoJSON")

Querying OSM for: Arapiraca, Alagoas, Brazil


## Grid size

Following Albors Zumel et al. (2025), each cell should cover ~0.077 sq. miles (~0.2 km²). `GRID_SIZE` is derived from the city's bounding box so the same block works for any city whose polygon is stored in `osm_arapiraca.geojson`.

## 3. Cálculo do tamanho do grid

Calcula N a partir da área do bounding box, visando células de ~0,2 km².

In [9]:
import math
import geopandas as gpd

TARGET_CELL_KM2 = 0.2  # 0.077 sq. miles

_mun = gpd.read_file("./output/arapiraca/osm_arapiraca.geojson").to_crs("EPSG:4326")
_gdf = _mun[_mun["name"] == "Arapiraca"]
_min_lon, _min_lat, _max_lon, _max_lat = _gdf.total_bounds
_lat_c = (_min_lat + _max_lat) / 2
_bbox_km2 = (
    (_max_lon - _min_lon) * 111.320 * math.cos(math.radians(_lat_c))
    * (_max_lat - _min_lat) * 110.574
)
GRID_SIZE = round(math.sqrt(_bbox_km2 / TARGET_CELL_KM2))
print(f"Arapiraca bbox: {_bbox_km2:.1f} km^2")
print(f"GRID_SIZE = {GRID_SIZE}  ({GRID_SIZE**2} cells of ~{TARGET_CELL_KM2} km^2)")

Arapiraca bbox: 638.5 km^2
GRID_SIZE = 57  (3249 cells of ~0.2 km^2)


## 4. Geração da máscara de validade

Cria uma matriz binária N×N indicando quais células do grid estão dentro do polígono municipal.

In [10]:
# Gera a máscara binária de validade
# Para cada célula do grid, verifica se há interseção com o polígono municipal
# 1 = célula válida (dentro da cidade), 0 = fora

import numpy as np
import geopandas as gpd
import shapely.geometry

grid_size = GRID_SIZE

mun = gpd.read_file("./output/arapiraca/osm_arapiraca.geojson")
mun = mun.to_crs("EPSG:4326")
gdf = mun[mun["name"] == "Arapiraca"].copy()
city_boundary = gdf.union_all()

bbox = gdf.total_bounds
min_lon, min_lat, max_lon, max_lat = bbox[2], bbox[1], bbox[0], bbox[3]

lon = np.linspace(min_lon, max_lon, grid_size + 1)
lat = np.linspace(min_lat, max_lat, grid_size + 1)

mask = np.zeros((grid_size, grid_size), dtype=np.float32)

for i in range(grid_size):
    for j in range(grid_size):
        cell_box = shapely.geometry.box(lon[j], lat[i], lon[j + 1], lat[i + 1])
        if city_boundary.intersects(cell_box):
            mask[i, j] = 1

np.save("./output/arapiraca/arapiraca_mask_final.npy", mask)
print(f"Mask shape: {mask.shape}, valid cells: {int(mask.sum())}/{grid_size**2}")

Mask shape: (57, 57), valid cells: 1896/3249


## 5. Filtragem espacial dos pontos de crime

Realiza um spatial join entre os pontos de ocorrência e o polígono municipal, mantendo apenas os registros dentro dos limites de Arapiraca.

In [11]:
# Spatial join: mantém apenas pontos dentro do polígono municipal
# Remove registros fora dos limites da cidade

# Spatial join points with municipality polygons (geojs-27-mun.json)
# Requires: geopandas
import geopandas as gpd

# load municipalities and ensure CRS is EPSG:4326
mun = gpd.read_file("./output/arapiraca/osm_arapiraca.geojson")
mun = mun.to_crs("EPSG:4326")

# create GeoDataFrame for Maceió points
gdf_arapiraca = gpd.GeoDataFrame(
    df_arapiraca.copy(),
    geometry=gpd.points_from_xy(df_arapiraca["LONGITUDE"], df_arapiraca["LATITUDE"]),
    crs="EPSG:4326",
)

# spatial join: attach municipality 'name' to each point (left join)
# use predicate='within' so points must fall inside polygon
joined = gpd.sjoin(
    gdf_arapiraca, mun[["name", "geometry"]], how="left", predicate="within"
)

# boolean: specifically in Maceió municipality
joined["in_arapiraca"] = joined["name"] == "Arapiraca"

# summary counts
total = len(df_arapiraca)
inside_count = int(joined["in_arapiraca"].sum())
outside_count = total - inside_count
print(f"Arapiraca records total: {total}")
print(f" - inside Arapiraca polygon: {inside_count}")
print(f" - outside Arapiraca polygon: {outside_count}")

# Filter df_arapiraca to only records whose points lie inside the Arapiraca polygon
inside_arapiraca = joined.loc[joined["in_arapiraca"]].copy()

inside_arapiraca_clean = inside_arapiraca.drop(
    columns=["geometry", "name", "in_arapiraca", "index_right"]
)

# reset index and overwrite df_arapiraca with filtered rows
df_arapiraca = inside_arapiraca_clean.reset_index(drop=True)

# save filtered CSV
output_path = "./output/arapiraca/filtered_arapiraca_inside_polygon.csv"
df_arapiraca.to_csv(output_path, index=False, sep=";")
print(f"Saved {len(df_arapiraca)} Arapiraca rows inside polygon to {output_path}")

Arapiraca records total: 17421
 - inside Arapiraca polygon: 17371
 - outside Arapiraca polygon: 50
Saved 17371 Arapiraca rows inside polygon to ./output/arapiraca/filtered_arapiraca_inside_polygon.csv


## 6. Visualização em mapa interativo

Exibe o polígono municipal e os pontos de crime (dentro/fora) em um mapa folium.

In [14]:
# Visualização: mapa interativo com polígono e pontos de crime

import folium

# Show the map with the polygon and the points inside/outside the polygon
osm = gpd.read_file("./output/arapiraca/osm_arapiraca.geojson").to_crs("EPSG:4326")
centroid = osm.geometry.centroid.iloc[0]
m = folium.Map(location=[centroid.y, centroid.x], zoom_start=11)

folium.GeoJson(
    osm.to_json(),
    name="Maceió (OSM)",
    style_function=lambda f: {
        "fillColor": "transparent",
        "color": "blue",
        "weight": 2,
        "opacity": 0.8,
    },
).add_to(m)

# optional: overlay sample points if gdf_arapiraca exists
if "gdf_arapiraca" in globals():
    for _, r in joined.iterrows():
        folium.CircleMarker(
            [r.geometry.y, r.geometry.x],
            radius=2,
            color="green" if r["in_arapiraca"] else "red",
            fill=True,
        ).add_to(m)
m.save("./output/arapiraca/arapiraca_and_points.html")
print("Saved arapiraca_osm_polygon_map.html")

/tmp/ipykernel_840132/1165562894.py:5: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroid = osm.geometry.centroid.iloc[0]


Saved arapiraca_osm_polygon_map.html
